In [15]:
import sys
sys.path.append("../")  # Adds the scripts folder to the Python path

In [ ]:
import pandas as pd

# Load final solar dataset
df = pd.read_csv(r"C:\Projects\GitHub\Ireland-energy-forecast\data\processed\Final_Solar_Data_Model.csv", parse_dates=["Date"], index_col="Date")

# Keep only valid years (2022–2024)
df = df.loc["2022-01-01":"2024-12-01"].copy()

# Define target & core features 
TARGET = "Solar_GWh"

# Core temporal lags
LAGS = ["Solar_GWh_Lag1", "Solar_GWh_Lag2", "Solar_GWh_Lag12"]

# Physics-driven features
CORE_FEATURES = ["Solar_Radiation_MJ_per_m2", "Solar_Capacity_MW"]

# Final feature set for Step 1
FEATURES = LAGS + CORE_FEATURES

# Drop NA
df = df[[TARGET] + FEATURES].dropna()

# split
train = df.loc["2022-01-01":"2023-12-01"]
test  = df.loc["2024-01-01":"2024-12-01"]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train period: {X_train.index.min().date()} → {X_train.index.max().date()}")
print(f"Test  period: {X_test.index.min().date()} → {X_test.index.max().date()}")

print("Train target zeros:", (y_train == 0).sum())
print("Test target zeros:", (y_test == 0).sum())

In [ ]:
import pandas as pd

# Paths for data, figures, and models
DATA_PATH  = r"C:\Projects\GitHub\Ireland-energy-forecast\data\processed\Final_Solar_Data_Model.csv"
FIG_DIR    = r"C:\Projects\GitHub\Ireland-energy-forecast\outputs\figures\NGBoost_Solar"
MODEL_PATH = r"C:\Projects\GitHub\Ireland-energy-forecast\models"

# Target and features
TARGET = "Solar_GWh"
FEATURES = [
    "Solar_GWh_Lag1", "Solar_GWh_Lag2", "Solar_GWh_Lag12",
    "Solar_Radiation_MJ_per_m2", "Solar_Capacity_MW"
]

# Load dataset and select the 2022–2024 window
df = pd.read_csv(DATA_PATH, parse_dates=["Date"], index_col="Date").sort_index()
df = df.loc["2022-01-01":"2024-12-01", [TARGET] + FEATURES].dropna()

# Define test set (fixed year 2024)
test = df.loc["2024-01-01":"2024-12-01"].copy()

# Define training set (2022–2023)
train = df.loc["2022-01-01":"2023-12-01"].copy()

# Drop months with zero generation from training
# Alternative approach: drop rows where installed capacity is zero
train = train[train[TARGET] > 0].copy()


# Split into features and target
X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

# Print summary
print(f"Train period: {train.index.min().date()} to {train.index.max().date()} | shape: {X_train.shape}")
print(f"Test period : {test.index.min().date()} to {test.index.max().date()} | shape: {X_test.shape}")
print("Train target zeros:", int((y_train == 0).sum()))
print("Test target zeros :", int((y_test == 0).sum()))


In [ ]:
# NGBoost on cleaned solar training data (drops early zeros)
# Uses expanding window cross-validation, trains final model, evaluates, and plots forecast vs actual

import os, time, joblib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paths
CSV_PATH   = r"C:\Projects\GitHub\Ireland-energy-forecast\data\processed\Final_Solar_Data_Model.csv"
FIG_DIR    = r"C:\Projects\GitHub\Ireland-energy-forecast\outputs\figures\NGBoost_Solar"
MODEL_PATH = r"C:\Projects\GitHub\Ireland-energy-forecast\models\NGBoost_Solar.joblib"
os.makedirs(FIG_DIR, exist_ok=True)

TARGET = "Solar_GWh"
EXOG   = ["Solar_GWh_Lag1", "Solar_GWh_Lag12", "Solar_Radiation_MJ_per_m2", "Solar_Capacity_MW"]

# Load dataset and prepare train/test sets
df = pd.read_csv(CSV_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date").sort_index()

# Drop early 2022 months with structural zeros
train = df.loc["2022-08-01":"2023-12-01"].copy()
test  = df.loc["2024-01-01":"2024-12-01"].copy()

# Keep only relevant columns and drop missing rows
cols_needed = [TARGET] + EXOG
train = train[cols_needed].dropna().copy()
test  = test[cols_needed].dropna().copy()

X_train, y_train = train[EXOG].astype(float), train[TARGET].astype(float)
X_test,  y_test  = test[EXOG].astype(float),  test[TARGET].astype(float)

print(f"Train period: {X_train.index.min().date()} to {X_train.index.max().date()} | n={len(X_train)}")
print(f"Test period : {X_test.index.min().date()} to {X_test.index.max().date()} | n={len(X_test)}")

# Expanding folds for time series cross-validation
warnings.filterwarnings("ignore")
from ngboost import NGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

def expanding_folds(X, y):
    cuts = [
        ("2022-08-01", "2023-04-01", "2023-05-01", "2023-08-01"),
        ("2022-08-01", "2023-08-01", "2023-09-01", "2023-10-01"),
        ("2022-08-01", "2023-10-01", "2023-11-01", "2023-12-01"),
    ]
    for tr_s, tr_e, va_s, va_e in cuts:
        Xtr, ytr = X.loc[tr_s:tr_e], y.loc[tr_s:tr_e]
        Xva, yva = X.loc[va_s:va_e], y.loc[va_s:va_e]
        if len(Xtr) > 0 and len(Xva) > 0:
            yield Xtr, ytr, Xva, yva

# Small hyperparameter grid
grid = [
    dict(n_estimators=150, learning_rate=0.03, max_depth=2, min_leaf=4),
    dict(n_estimators=250, learning_rate=0.03, max_depth=2, min_leaf=6),
    dict(n_estimators=300, learning_rate=0.05, max_depth=3, min_leaf=6),
]

best_cfg, best_mae = None, np.inf
for cfg in grid:
    maes = []
    for Xtr, ytr, Xva, yva in expanding_folds(X_train, y_train):
        base = DecisionTreeRegressor(max_depth=cfg["max_depth"], min_samples_leaf=cfg["min_leaf"], random_state=42)
        ngb  = NGBRegressor(
            Base=base,
            n_estimators=cfg["n_estimators"],
            learning_rate=cfg["learning_rate"],
            natural_gradient=True,
            verbose=False,
            random_state=42
        )
        ngb.fit(Xtr, ytr)
        pred = ngb.predict(Xva)
        maes.append(mean_absolute_error(yva, pred))
    mean_mae = float(np.mean(maes))
    print(f"CV {cfg} | MAE {mean_mae:.2f}")
    if mean_mae < best_mae:
        best_mae, best_cfg = mean_mae, cfg

print(f"\nBest NGBoost config (CV MAE {best_mae:.2f}): {best_cfg}")

# Train final NGBoost model on the clean training set
base = DecisionTreeRegressor(max_depth=best_cfg["max_depth"], min_samples_leaf=best_cfg["min_leaf"], random_state=42)
ngb  = NGBRegressor(
    Base=base,
    n_estimators=best_cfg["n_estimators"],
    learning_rate=best_cfg["learning_rate"],
    natural_gradient=True,
    verbose=True,
    random_state=42
)

t0 = time.time()
ngb.fit(X_train, y_train)
train_time = time.time() - t0

# Evaluate on 2024
from sklearn.metrics import mean_absolute_error, r2_score

pred_mean = ngb.predict(X_test)
mae  = mean_absolute_error(y_test, pred_mean)
rmse = float(np.sqrt(np.mean((y_test - pred_mean) ** 2)))
r2   = r2_score(y_test, pred_mean)

print(f"\nTest MAE {mae:.2f} | RMSE {rmse:.2f} | R² {r2:.3f}")

# Save model
joblib.dump(ngb, MODEL_PATH)
print("Saved:", MODEL_PATH)

# Plot forecast vs actual with 90% confidence intervals
pred_dist = ngb.pred_dist(X_test)
q05 = pred_dist.ppf(0.05)
q95 = pred_dist.ppf(0.95)

plt.figure(figsize=(10, 5))
plt.plot(y_test.index, y_test.values, label="Actual", color="#000000", linewidth=2)
plt.plot(y_test.index, pred_mean, label="NGBoost Forecast", color="#1f77b4", linestyle="--", linewidth=2)
plt.fill_between(y_test.index, q05, q95, color="#aec7e8", alpha=0.18, label="90% CI")
plt.title("NGBoost Solar Forecast vs Actual (2024)")
plt.xlabel("Date"); plt.ylabel("Solar Generation (GWh)")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, "ngboost_solar_operational_forecast_2024.png")
plt.savefig(fig_path, dpi=300)
plt.show()
print("Figure saved to:", fig_path)


In [ ]:
# ---- Log Model Performance (NGBoost Solar Operational) ----
from log_utils import log_model_performance, save_log_to_csv

log_dict = log_model_performance(
    model_name="NGBoost_Solar",
    target_variable=TARGET,
    X_train=X_train,
    X_test=X_test,
    y_test=y_test.values,
    y_pred=pred_mean,
    model_object=ngb,
    training_time_sec=train_time,
    dataset_name=os.path.basename(CSV_PATH),
    tuning_type=f"Manual CV (expanding folds) → {best_cfg}",
    gpu_used="NVIDIA GeForce RTX 2050",
    prediction_interval_width=float(np.mean(q95 - q05)),
    extra_notes=f"Train: 2022-08 to 2023-12 (zeros dropped); Test: 2024 full year; Features={EXOG}"
)

save_log_to_csv(log_dict)


In [ ]:
# NGBoost (Solar) Forecast vs Actual using a saved model

import os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# File paths
CSV_PATH   = r"C:\Projects\GitHub\Ireland-energy-forecast\data\processed\Final_Solar_Data_Model.csv"
MODEL_PATH = r"C:\Projects\GitHub\Ireland-energy-forecast\models\NGBoost_Solar.joblib"
FIG_DIR    = r"C:\Projects\GitHub\Ireland-energy-forecast\outputs\figures\NGBoost_Solar"
os.makedirs(FIG_DIR, exist_ok=True)
FIG_PATH   = os.path.join(FIG_DIR, "ngboost_solar_operational_forecast_2024_recolored.png")

# Target and features (must match what the model was trained on)
TARGET = "Solar_GWh"
EXOG   = ["Solar_GWh_Lag1", "Solar_GWh_Lag12", "Solar_Radiation_MJ_per_m2", "Solar_Capacity_MW"]

# Load data and select the 2024 test window
df = pd.read_csv(CSV_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
test = df.loc["2024-01-01":"2024-12-01", [TARGET] + EXOG].dropna()

X_test = test[EXOG].astype(float)
y_test = test[TARGET].astype(float)

# Load the trained NGBoost model
ngb = joblib.load(MODEL_PATH)

# Predict mean values and 90% prediction intervals
pred_mean = ngb.predict(X_test)
dist      = ngb.pred_dist(X_test)
q05, q95  = dist.ppf(0.05), dist.ppf(0.95)

# Plot actual vs forecast with prediction intervals
plt.figure(figsize=(10, 5))
plt.plot(y_test.index, y_test.values, label="Actual", color="black", linewidth=2)
plt.plot(y_test.index, pred_mean, label="NGBoost Forecast", color="#1f77b4", linestyle="--", linewidth=2)
plt.fill_between(y_test.index, q05, q95, color="#aec7e8", alpha=0.3, label="90% CI")

plt.title("NGBoost Solar Forecast vs Actual (2024)")
plt.xlabel("Date")
plt.ylabel("Solar Generation (GWh)")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved to:", FIG_PATH)


In [ ]:
# Compute CRPS table for NGBoost Solar model

import os
import numpy as np
import pandas as pd
import joblib
from scipy.stats import norm

# File paths
PROJECT_ROOT = r"C:\Projects\GitHub\Ireland-energy-forecast"
CSV_PATH     = os.path.join(PROJECT_ROOT, "data", "processed", "Final_Solar_Data_Model.csv")
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models", "NGBoost_Solar.joblib")
OUT_CSV      = os.path.join(PROJECT_ROOT, "outputs", "results_tables", "NGBoost_Solar_CRPS.csv")

# Target and features (must match model training)
TARGET = "Solar_GWh"
EXOG   = ["Solar_GWh_Lag1", "Solar_GWh_Lag12", "Solar_Radiation_MJ_per_m2", "Solar_Capacity_MW"]

# Load dataset and restrict to training and test periods
df = pd.read_csv(CSV_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
cols_needed = [TARGET] + EXOG
df = df[cols_needed].dropna().copy()

train = df.loc["2022-08-01":"2023-12-01"].copy()
test  = df.loc["2024-01-01":"2024-12-01"].copy()

X_train, y_train = train[EXOG].astype(float), train[TARGET].astype(float)
X_test,  y_test  = test[EXOG].astype(float),  test[TARGET].astype(float)

print(f"Train period: {X_train.index.min().date()} to {X_train.index.max().date()} | n={len(X_train)}")
print(f"Test period : {X_test.index.min().date()} to {X_test.index.max().date()} | n={len(X_test)}")

# Load trained NGBoost model
ngb = joblib.load(MODEL_PATH)

# Predictive distribution for the test set
dist = ngb.pred_dist(X_test)

# Extract mean and standard deviation safely
mu = getattr(dist, "loc", None)
sigma = getattr(dist, "scale", None)

if mu is None or sigma is None:
    params = getattr(dist, "params", None)
    if params is not None:
        if isinstance(params, dict):
            mu = params.get("loc", params.get("mean", None))
            sigma = params.get("scale", params.get("std", None))
        else:
            try:
                mu, sigma = params
            except Exception:
                pass

if mu is None:
    try: mu = dist.mean()
    except Exception: pass

if sigma is None:
    try: sigma = dist.std()
    except Exception: pass

mu = np.asarray(mu, dtype=float)
sigma = np.asarray(sigma, dtype=float)
sigma = np.maximum(sigma, 1e-8)  # prevent zero or negative values

# Compute CRPS for a Normal distribution
# CRPS(N(μ,σ), y) = σ * [1/√π - 2φ(z) - z(2Φ(z)-1)]
y_true = y_test.values.astype(float)
z = (y_true - mu) / sigma
phi = norm.pdf(z)
Phi = norm.cdf(z)
crps = sigma * ((1.0 / np.sqrt(np.pi)) - 2.0 * phi - z * (2.0 * Phi - 1.0))

# Save results as a table
crps_df = pd.DataFrame({
    "Date": X_test.index,
    "Actual": y_true,
    "Predicted_Mean": mu,
    "Predicted_Std": sigma,
    "CRPS": crps
}).set_index("Date")

os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
crps_df.to_csv(OUT_CSV)

print(f"CRPS table saved to {OUT_CSV}")
print(f"Mean CRPS: {crps_df['CRPS'].mean():.4f}")


In [ ]:
# SHAP analysis for NGBoost (beeswarm, bar, waterfall)

import os, json
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import joblib

FIG_DIR = r"C:\Projects\GitHub\Ireland-energy-forecast\outputs\figures\NGBoost_Solar"
os.makedirs(FIG_DIR, exist_ok=True)

# Load model if needed
MODEL_PATH = r"C:\Projects\GitHub\Ireland-energy-forecast\models\NGBoost_Solar.joblib"
ngb = joblib.load(MODEL_PATH)

# Use a small background sample for speed and stability
bg_n = min(200, len(X_train))
background = X_train.sample(bg_n, random_state=42).copy()

# SHAP needs a callable that takes ndarray and returns predictions
def ngb_predict_nd(X_nd):
    X_df = pd.DataFrame(X_nd, columns=EXOG)
    return ngb.predict(X_df)

# Build explainer with an independent masker (PermutationExplainer under the hood)
masker = shap.maskers.Independent(background)
explainer = shap.Explainer(ngb_predict_nd, masker)

# Compute SHAP values on the test set
shap_values = explainer(X_test.values)
shap_values.feature_names = EXOG
shap_values.data = X_test.values
shap_values.base_values = np.atleast_1d(shap_values.base_values)

# Beeswarm plot
plt.figure(figsize=(9, 6))
shap.plots.beeswarm(shap_values, show=False, max_display=len(EXOG))
plt.title("NGBoost (Solar) — SHAP Beeswarm (test set)")
beeswarm_path = os.path.join(FIG_DIR, "ngboost_solar_shap_beeswarm.png")
plt.tight_layout()
plt.savefig(beeswarm_path, dpi=300)
plt.show()

# Bar plot (mean absolute SHAP)
plt.figure(figsize=(9, 6))
shap.plots.bar(shap_values, show=False, max_display=len(EXOG))
plt.title("NGBoost (Solar) — Mean |SHAP| (feature importance)")
bar_path = os.path.join(FIG_DIR, "ngboost_solar_shap_bar.png")
plt.tight_layout()
plt.savefig(bar_path, dpi=300)
plt.show()

# Waterfall plot for a representative month (median absolute error)
abs_err = np.abs(y_test.values - ngb.predict(X_test))
idx_med = int(np.argsort(abs_err)[len(abs_err) // 2])

plt.figure(figsize=(9, 6))
shap.plots.waterfall(shap_values[idx_med], show=False, max_display=len(EXOG))
plt.title(f"NGBoost (Solar) — SHAP Waterfall (test idx={idx_med}, date={X_test.index[idx_med].date()})")
waterfall_path = os.path.join(FIG_DIR, f"ngboost_solar_shap_waterfall_idx{idx_med}.png")
plt.tight_layout()
plt.savefig(waterfall_path, dpi=300)
plt.show()

print("Saved SHAP plots:")
print(" -", beeswarm_path)
print(" -", bar_path)
print(" -", waterfall_path)
